In [18]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/abidrahman25/loan-prediction-dataset-ctec3702/Part B - Loan Prediction Dataset/train_loan.csv


In [19]:
!pip install -q pyspark

In [20]:
from pyspark.sql import SparkSession



spark = SparkSession.builder.appName("CTEC3702_PartB").getOrCreate()

spark.sparkContext.setLogLevel("ERROR")



print("✅ Spark started! Version:", spark.version)

print("✅ Spark running:", spark.sparkContext is not None)

✅ Spark started! Version: 4.0.1
✅ Spark running: True


In [21]:
import glob



csv_files = glob.glob("/kaggle/input/**/**.csv", recursive=True)

print("Found CSVs:", len(csv_files))

csv_files[:10]

Found CSVs: 1


['/kaggle/input/datasets/abidrahman25/loan-prediction-dataset-ctec3702/Part B - Loan Prediction Dataset/train_loan.csv']

In [22]:
CSV_PATH = "/kaggle/input/datasets/abidrahman25/loan-prediction-dataset-ctec3702/Part B - Loan Prediction Dataset/train_loan.csv"



df_raw = spark.read.csv(CSV_PATH, header=True, inferSchema=True)

df_raw.printSchema()

df_raw.show(5, truncate=False)

print("Rows:", df_raw.count())

root
 |-- Loan_ID: string (nullable = true)
 |-- Gender: string (nullable = true)
 |-- Married: string (nullable = true)
 |-- Dependents: string (nullable = true)
 |-- Education: string (nullable = true)
 |-- Self_Employed: string (nullable = true)
 |-- ApplicantIncome: integer (nullable = true)
 |-- CoapplicantIncome: double (nullable = true)
 |-- LoanAmount: integer (nullable = true)
 |-- Loan_Amount_Term: integer (nullable = true)
 |-- Credit_History: integer (nullable = true)
 |-- Property_Area: string (nullable = true)
 |-- Loan_Status: string (nullable = true)

+--------+------+-------+----------+------------+-------------+---------------+-----------------+----------+----------------+--------------+-------------+-----------+
|Loan_ID |Gender|Married|Dependents|Education   |Self_Employed|ApplicantIncome|CoapplicantIncome|LoanAmount|Loan_Amount_Term|Credit_History|Property_Area|Loan_Status|
+--------+------+-------+----------+------------+-------------+---------------+-------------

In [23]:
from pyspark.sql import functions as F



df = df_raw



# Clean Dependents column (replace 3+ with 3)

df = df.withColumn(

    "Dependents",

    F.when(F.col("Dependents") == "3+", "3").otherwise(F.col("Dependents"))

)



print("Columns:", df.columns)



# simple preview

df.show(5)

Columns: ['Loan_ID', 'Gender', 'Married', 'Dependents', 'Education', 'Self_Employed', 'ApplicantIncome', 'CoapplicantIncome', 'LoanAmount', 'Loan_Amount_Term', 'Credit_History', 'Property_Area', 'Loan_Status']
+--------+------+-------+----------+------------+-------------+---------------+-----------------+----------+----------------+--------------+-------------+-----------+
| Loan_ID|Gender|Married|Dependents|   Education|Self_Employed|ApplicantIncome|CoapplicantIncome|LoanAmount|Loan_Amount_Term|Credit_History|Property_Area|Loan_Status|
+--------+------+-------+----------+------------+-------------+---------------+-----------------+----------+----------------+--------------+-------------+-----------+
|LP001002|  Male|     No|         0|    Graduate|           No|           5849|              0.0|      NULL|             360|             1|        Urban|          Y|
|LP001003|  Male|    Yes|         1|    Graduate|           No|           4583|           1508.0|       128|             3

In [24]:
# Create label column

df = df.withColumn(

    "label",

    F.when(F.col("Loan_Status") == "Y", 1.0).otherwise(0.0)

)



# Identify categorical and numeric columns

categorical_cols = [c for c, t in df.dtypes if t == "string" and c not in ["Loan_Status", "Loan_ID"]]

numeric_cols = [c for c, t in df.dtypes if t in ["int", "double"] and c != "label"]



print("Categorical columns:", categorical_cols)

print("Numeric columns:", numeric_cols)

Categorical columns: ['Gender', 'Married', 'Dependents', 'Education', 'Self_Employed', 'Property_Area']
Numeric columns: ['ApplicantIncome', 'CoapplicantIncome', 'LoanAmount', 'Loan_Amount_Term', 'Credit_History']


In [25]:
from pyspark.sql import functions as F



# Fill missing categoricals

df = df.fillna("Unknown", subset=categorical_cols)



# Fill missing numerics with mean

for c in numeric_cols:

    mean_val = df.select(F.mean(F.col(c))).first()[0]

    if mean_val is not None:

        df = df.fillna({c: float(mean_val)})



print("✅ Missing values handled")

✅ Missing values handled


In [26]:
from pyspark.ml import Pipeline

from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler



indexers = [StringIndexer(inputCol=c, outputCol=f"{c}_idx", handleInvalid="keep") for c in categorical_cols]

encoders = [OneHotEncoder(inputCols=[f"{c}_idx"], outputCols=[f"{c}_ohe"]) for c in categorical_cols]



feature_cols = [f"{c}_ohe" for c in categorical_cols] + numeric_cols



assembler = VectorAssembler(inputCols=feature_cols, outputCol="features", handleInvalid="keep")



pipeline = Pipeline(stages=indexers + encoders + [assembler])

pipeline_model = pipeline.fit(df)



df_vec = pipeline_model.transform(df).select("features", "label")



df_vec.show(3, truncate=False)

print("✅ df_vec created")



+--------------------------------------------------------------------------------------------+-----+
|features                                                                                    |label|
+--------------------------------------------------------------------------------------------+-----+
|(24,[0,4,6,11,13,17,19,21,22,23],[1.0,1.0,1.0,1.0,1.0,1.0,5849.0,146.0,360.0,1.0])          |1.0  |
|(24,[0,3,7,11,13,18,19,20,21,22,23],[1.0,1.0,1.0,1.0,1.0,1.0,4583.0,1508.0,128.0,360.0,1.0])|0.0  |
|(24,[0,3,6,11,14,17,19,21,22,23],[1.0,1.0,1.0,1.0,1.0,1.0,3000.0,66.0,360.0,1.0])           |1.0  |
+--------------------------------------------------------------------------------------------+-----+
only showing top 3 rows
✅ df_vec created


In [27]:
train_df, test_df = df_vec.randomSplit([0.72, 0.28], seed=42)

print("Train rows:", train_df.count(), "| Test rows:", test_df.count())

Train rows: 468 | Test rows: 146


In [28]:
from pyspark.ml.classification import LogisticRegression, DecisionTreeClassifier, RandomForestClassifier

from pyspark.ml.evaluation import MulticlassClassificationEvaluator



acc_eval = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="accuracy")



# Logistic Regression

lr = LogisticRegression(featuresCol="features", labelCol="label")

lr_model = lr.fit(train_df)

lr_pred = lr_model.transform(test_df)

lr_acc = acc_eval.evaluate(lr_pred)



# Decision Tree

dt = DecisionTreeClassifier(featuresCol="features", labelCol="label", maxDepth=5, seed=42)

dt_model = dt.fit(train_df)

dt_pred = dt_model.transform(test_df)

dt_acc = acc_eval.evaluate(dt_pred)



# Random Forest

rf = RandomForestClassifier(featuresCol="features", labelCol="label", numTrees=100, seed=42)

rf_model = rf.fit(train_df)

rf_pred = rf_model.transform(test_df)

rf_acc = acc_eval.evaluate(rf_pred)



print(f"✅ Accuracy — LR: {lr_acc:.3f} | DT: {dt_acc:.3f} | RF: {rf_acc:.3f}")

✅ Accuracy — LR: 0.753 | DT: 0.740 | RF: 0.747


In [29]:
from pyspark.ml.evaluation import MulticlassClassificationEvaluator



precision_eval = MulticlassClassificationEvaluator(

    labelCol="label",

    predictionCol="prediction",

    metricName="weightedPrecision"

)



recall_eval = MulticlassClassificationEvaluator(

    labelCol="label",

    predictionCol="prediction",

    metricName="weightedRecall"

)



# Logistic Regression metrics

lr_precision = precision_eval.evaluate(lr_pred)

lr_recall = recall_eval.evaluate(lr_pred)



# Decision Tree metrics

dt_precision = precision_eval.evaluate(dt_pred)

dt_recall = recall_eval.evaluate(dt_pred)



# Random Forest metrics

rf_precision = precision_eval.evaluate(rf_pred)

rf_recall = recall_eval.evaluate(rf_pred)



print("LR Precision:", lr_precision, "| Recall:", lr_recall)

print("DT Precision:", dt_precision, "| Recall:", dt_recall)

print("RF Precision:", rf_precision, "| Recall:", rf_recall)

LR Precision: 0.7429995970991137 | Recall: 0.7534246575342466
DT Precision: 0.7250961307378034 | Recall: 0.7397260273972603
RF Precision: 0.7341165320314307 | Recall: 0.7465753424657535


In [30]:
# Confusion matrices

print("LR Confusion Matrix")

lr_pred.groupBy("label","prediction").count().show()



print("DT Confusion Matrix")

dt_pred.groupBy("label","prediction").count().show()



print("RF Confusion Matrix")

rf_pred.groupBy("label","prediction").count().show()

LR Confusion Matrix
+-----+----------+-----+
|label|prediction|count|
+-----+----------+-----+
|  1.0|       1.0|   91|
|  0.0|       1.0|   21|
|  1.0|       0.0|   15|
|  0.0|       0.0|   19|
+-----+----------+-----+

DT Confusion Matrix
+-----+----------+-----+
|label|prediction|count|
+-----+----------+-----+
|  1.0|       1.0|   91|
|  0.0|       1.0|   23|
|  1.0|       0.0|   15|
|  0.0|       0.0|   17|
+-----+----------+-----+

RF Confusion Matrix
+-----+----------+-----+
|label|prediction|count|
+-----+----------+-----+
|  1.0|       1.0|   91|
|  0.0|       1.0|   22|
|  1.0|       0.0|   15|
|  0.0|       0.0|   18|
+-----+----------+-----+

